# Fase 3 — Núcleo algorítmico, eficiencia y POO
## Recomendador híbrido de películas TMDB 2010–2025 · Grupo 4

Esta fase convierte los pasos sueltos de la Fase 2 en clases reutilizables (módulos en `src/`) y mide su costo. Se sigue la estructura del proyecto modelo de la Semana 2 (`Material_Didactico_Semana2`), adaptada al caso TMDB.

**Qué se entrega a la Fase 4:** el conjunto procesado, el pipeline con sus parámetros aprendidos y el esquema final, todo en `data/processed/`.

## Configuración
Única celda a editar. Todo lo demás se construye desde ella.

In [1]:
import sys
from pathlib import Path

# Localiza la raíz del proyecto aunque Jupyter se abra en otra carpeta
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "pipeline.py").exists())
sys.path.insert(0, str(RAIZ))

SEMILLA = 42
RUTA_DATOS = RAIZ / "data" / "raw" / "movies_raw.csv"
DIR_PROCESADO = RAIZ / "data" / "processed"
FRACCION_PRUEBA = 0.20

COLUMNAS_ESPERADAS = [
    "movie_id", "title", "release_date", "status", "budget", "revenue", "runtime",
    "genres", "production_companies", "production_countries", "spoken_languages",
    "original_language", "vote_average", "vote_count", "popularity", "adult",
]
COLUMNAS_NUMERICAS = ["budget", "revenue", "runtime", "vote_average", "vote_count", "popularity"]
COLUMNAS_A_ESCALAR = ["vote_average", "vote_count", "popularity"]
COLUMNA_SESGADA = "popularity"   # log1p antes de escalar: cola larga a la derecha

## 1. Carga con verificación de esquema
`src/carga.py` falla con un mensaje claro si falta el archivo o una columna declarada.

In [2]:
import pandas as pd
from src.carga import cargar

df = cargar(RUTA_DATOS, COLUMNAS_ESPERADAS, COLUMNAS_NUMERICAS)
print(f"{df.shape[0]:,} filas x {df.shape[1]} columnas")
df[["title", "genres", "production_companies", "vote_average", "vote_count", "popularity"]].head()

17,978 filas x 16 columnas


,title,genres,production_companies,vote_average,vote_count,popularity
0,Dark Seduction,Drama|Thriller|Romance,Lionsgate,6.799,102,1.6261
1,12,Comedy,NaN,5.000,57,0.3124
2,"Lula, the Son of Brazil",Drama|History,LC Barreto|Filmes do Equador|InterVideo Produc...,5.500,90,1.3337
3,Protect and Serve,Comedy|Crime|Action,M6 Films|Same Player|Pathé|SCOPE Pictures|Webe...,4.500,106,0.5622
4,Re-Cut,Horror|Thriller,Fritz Manger Productions|National Picture Show,6.535,115,0.7341


## 2. Separación entrenamiento / prueba
Se separa **antes** de ajustar: los parámetros (mínimo y rango) se aprenden solo con entrenamiento y se aplican a prueba. Así no hay fuga de datos.

In [3]:
prueba = df.sample(frac=FRACCION_PRUEBA, random_state=SEMILLA)
entrenamiento = df.drop(prueba.index)
print(f"entrenamiento: {len(entrenamiento):,} | prueba: {len(prueba):,}")

entrenamiento: 14,382 | prueba: 3,596


## 3. Pipeline con clases
Todos los pasos heredan de `Transformador` (contrato `ajustar` / `transformar`, estado encapsulado, copia defensiva). El `Pipeline` los encadena y exige ajustar antes de transformar. Los pasos propios de TMDB (`ConstructorSopa`, `LogNaturalPlus1`, `EscaladorMinMax`) reproducen el Paso 2 y el Paso 5 de la Fase 2.

In [4]:
from src.pipeline import Pipeline
from src.transformadores import ConstructorSopa, LogNaturalPlus1, EscaladorMinMax


def construir_pipeline():
    pasos = [ConstructorSopa(), LogNaturalPlus1(COLUMNA_SESGADA)]
    pasos += [EscaladorMinMax(c) for c in COLUMNAS_A_ESCALAR]
    return Pipeline(pasos)


pipe = construir_pipeline()
pipe.ajustar(entrenamiento)
entrenamiento_final = pipe.transformar(entrenamiento)
prueba_final = pipe.transformar(prueba)
print(pipe)
pipe.resumen()

Pipeline(5 pasos, ajustado)


,orden,paso,clase
0,1,ConstructorSopa(metadata_soup),ConstructorSopa
1,2,LogNaturalPlus1(popularity),LogNaturalPlus1
2,3,EscaladorMinMax(vote_average),EscaladorMinMax
3,4,EscaladorMinMax(vote_count),EscaladorMinMax
4,5,EscaladorMinMax(popularity),EscaladorMinMax


In [5]:
# Los parámetros son los de entrenamiento: en prueba el rango puede salirse de [0, 1]
for c in COLUMNAS_A_ESCALAR:
    print(f"{c:13s} entrenamiento [{entrenamiento_final[c].min():.3f}, {entrenamiento_final[c].max():.3f}]"
          f" | prueba [{prueba_final[c].min():.3f}, {prueba_final[c].max():.3f}]")
entrenamiento_final[["title", "metadata_soup", *COLUMNAS_A_ESCALAR]].head()

vote_average  entrenamiento [0.000, 1.000] | prueba [0.056, 0.870]
vote_count    entrenamiento [0.000, 1.000] | prueba [0.000, 1.040]
popularity    entrenamiento [0.000, 1.000] | prueba [0.007, 0.970]


,title,metadata_soup,vote_average,vote_count,popularity
1,12,comedy en,0.392662,0.000181,0.056836
2,"Lula, the Son of Brazil",drama history lcbarreto filmesdoequador interv...,0.454636,0.001032,0.180760
4,Re-Cut,horror thriller fritzmangerproductions nationa...,0.582920,0.001677,0.116824
6,The Whistler,comedy m6films lesproductionsdutrésor europaco...,0.380268,0.000748,0.200080
7,A Serbian Film,crime horror thriller contrafilm sr,0.449058,0.056393,0.590673


## 4. Equivalencia con la Fase 2
El pipeline con clases debe dar **el mismo resultado** que el código de celdas sueltas de la Fase 2. Si no coincide, la refactorización cambió el comportamiento.

In [6]:
import numpy as np

# Versión Fase 2: celdas sueltas, sobre el mismo conjunto de entrenamiento
f2 = entrenamiento.copy()
f2["metadata_soup"] = (
    f2["genres"].fillna("").str.replace("|", " ", regex=False).str.lower() + " "
    + f2["production_companies"].fillna("").map(
        lambda t: " ".join(p.replace(" ", "").lower() for p in t.split("|")[:3] if p))
    + " " + f2["original_language"].fillna("").str.lower()
).str.split().str.join(" ")
f2["popularity"] = np.log1p(f2["popularity"])
for c in COLUMNAS_A_ESCALAR:
    f2[c] = (f2[c] - f2[c].min()) / ((f2[c].max() - f2[c].min()) or 1.0)

pd.testing.assert_frame_equal(f2, entrenamiento_final)
print("OK: el pipeline con clases reproduce la Fase 2 exactamente.")

OK: el pipeline con clases reproduce la Fase 2 exactamente.


## 5. Validación: caso normal, casos límite y excepciones

In [7]:
# --- CASO NORMAL ---
assert entrenamiento_final["metadata_soup"].str.len().gt(0).mean() > 0.99
assert entrenamiento_final[COLUMNAS_A_ESCALAR].min().min() >= 0
assert entrenamiento_final[COLUMNAS_A_ESCALAR].max().max() <= 1
print("Normal: sopa no vacía y variables escaladas en [0, 1].")

# --- CASOS LÍMITE ---
minimo = pd.DataFrame({
    "genres": [None], "production_companies": [None], "original_language": [None],
    "vote_average": [7.0], "vote_count": [100], "popularity": [0.0],
})
salida = construir_pipeline().ajustar(minimo).transformar(minimo)
assert salida["metadata_soup"].iloc[0] == ""      # todo nulo -> sopa vacía, sin error
assert salida["vote_average"].iloc[0] == 0.0      # rango 0 -> el divisor cae a 1.0, sin división por cero
print("Límite: una sola fila con nulos y columnas constantes no rompe el pipeline.")

# --- EXCEPCIONES ---
def debe_fallar(tipo, accion):
    try:
        accion()
    except tipo as e:
        print(f"OK {tipo.__name__}: {e}")
    else:
        raise AssertionError(f"Debía lanzar {tipo.__name__}")

debe_fallar(RuntimeError, lambda: construir_pipeline().transformar(entrenamiento))   # sin ajustar
debe_fallar(KeyError, lambda: construir_pipeline().ajustar(entrenamiento.drop(columns=["genres"])))
debe_fallar(TypeError, lambda: Pipeline().agregar("no soy un transformador"))
debe_fallar(ValueError, lambda: LogNaturalPlus1("popularity").ajustar_transformar(
    pd.DataFrame({"popularity": [-1.0, 2.0]})))
debe_fallar(FileNotFoundError, lambda: cargar(Path("data/raw/no_existe.csv"), COLUMNAS_ESPERADAS))

Normal: sopa no vacía y variables escaladas en [0, 1].
Límite: una sola fila con nulos y columnas constantes no rompe el pipeline.
OK RuntimeError: Pipeline: hay que ajustar antes de transformar.
OK KeyError: "ConstructorSopa(metadata_soup): faltan columnas de origen ['genres']."
OK TypeError: Se esperaba un Transformador y se recibió str.
OK ValueError: LogNaturalPlus1(popularity): log1p no admite valores negativos.
OK FileNotFoundError: No se encontró el archivo: data\raw\no_existe.csv


## 6. Eficiencia: tiempo y memoria
Se compara la construcción de la sopa con un bucle por fila (`iterrows`) frente a la versión por columnas. Se exige el **mismo resultado** antes de comparar y se conserva el menor tiempo de varias repeticiones. El tiempo y la memoria se miden por separado porque `tracemalloc` (memoria) ralentiza la ejecución.

In [8]:
import time

from src.medicion import medir


def sopa_bucle(df):
    filas = []
    for _, fila in df.iterrows():
        g = (fila["genres"] if isinstance(fila["genres"], str) else "").replace("|", " ").lower()
        prods = fila["production_companies"] if isinstance(fila["production_companies"], str) else ""
        pr = " ".join(p.replace(" ", "").lower() for p in prods.split("|")[:3] if p)
        idi = (fila["original_language"] if isinstance(fila["original_language"], str) else "").lower()
        filas.append(" ".join((g + " " + pr + " " + idi).split()))
    return pd.Series(filas, index=df.index, name="metadata_soup")


def sopa_columnas(df):
    return ConstructorSopa().ajustar_transformar(df)["metadata_soup"]


def tiempo_minimo(funcion, df, repeticiones=3):
    # tracemalloc ralentiza el código: el tiempo se mide aparte, sin él
    tiempos = []
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        resultado = funcion(df)
        tiempos.append(time.perf_counter() - inicio)
    return resultado, min(tiempos)


r_bucle, t_bucle = tiempo_minimo(sopa_bucle, entrenamiento)
r_cols, t_cols = tiempo_minimo(sopa_columnas, entrenamiento)
pd.testing.assert_series_equal(r_bucle, r_cols, check_dtype=False)   # mismo contenido o la comparación no vale

_, _, m_bucle = medir(sopa_bucle, entrenamiento)
_, _, m_cols = medir(sopa_columnas, entrenamiento)

pd.DataFrame({
    "versión": ["bucle iterrows", "por columnas (clase)"],
    "tiempo_s": [t_bucle, t_cols],
    "memoria_pico_MB": [m_bucle, m_cols],
}).round(3)

,versión,tiempo_s,memoria_pico_MB
0,bucle iterrows,0.297,4.780
1,por columnas (clase),0.036,12.629


In [9]:
relacion = t_bucle / t_cols
print(f"Tiempo: la versión por columnas es {relacion:.1f}x {'más rápida' if relacion >= 1 else 'más lenta'} que el bucle.")
print(f"Memoria pico: {m_bucle:.1f} MB (bucle) vs {m_cols:.1f} MB (columnas).")

Tiempo: la versión por columnas es 8.3x más rápida que el bucle.
Memoria pico: 4.8 MB (bucle) vs 12.6 MB (columnas).


## 7. Guardar el resultado y su configuración
El conjunto procesado no sirve sin el registro de cómo se obtuvo: se guardan también los parámetros aprendidos y la semilla.

In [10]:
import json

DIR_PROCESADO.mkdir(parents=True, exist_ok=True)
COLUMNAS_SALIDA = ["movie_id", "title", "metadata_soup", *COLUMNAS_A_ESCALAR]

entrenamiento_final[COLUMNAS_SALIDA].to_csv(DIR_PROCESADO / "entrenamiento_f3.csv", index=False)
prueba_final[COLUMNAS_SALIDA].to_csv(DIR_PROCESADO / "prueba_f3.csv", index=False)

parametros = {
    "semilla": SEMILLA,
    "fraccion_prueba": FRACCION_PRUEBA,
    "pasos": {p.nombre: p.parametros for p in pipe.pasos_ejecutados()},
}
(DIR_PROCESADO / "parametros_f3.json").write_text(
    json.dumps(parametros, indent=2, ensure_ascii=False), encoding="utf-8")

# Comprobación: lo guardado se vuelve a leer sin sorpresas
leido = pd.read_csv(DIR_PROCESADO / "entrenamiento_f3.csv")
assert list(leido.columns) == COLUMNAS_SALIDA and len(leido) == len(entrenamiento_final)
print("Guardado:", sorted(p.name for p in DIR_PROCESADO.glob("*_f3.*")))

Guardado: ['entrenamiento_f3.csv', 'parametros_f3.json', 'prueba_f3.csv']
